# Phase 3c — Fixing three problems the last run exposed

The paired test and the sweeps turned up three issues. All are fixable, and the third
is a better result than the one we were originally chasing.

**1. Physics is not free after all.** On identical splits:

| Regime | unconstrained | constrained | paired diff | p |
|---|---|---|---|---|
| A | 0.8340 | 0.8320 | −0.0021 | **0.005** |
| B | 0.7857 | 0.7775 | −0.0082 | **0.001** |
| C* | 0.6506 | 0.6443 | −0.0063 | 0.31 |

Small, but real and consistent. The claim becomes "physics costs ~0.005 ROC-AUC",
not "physics is free". That is still an easy trade for guaranteed consistency, but it
has to be stated honestly.

**2. The sweeps ran off the end of the data.** Only 2% of records sit above 1000 mM
NaCl, yet the salt sweep went to 2000 mM. Above ~400 mM the trees have no splits, so
the curves are flat by construction — and `argmax` on a flat curve returns an
arbitrary interior point. That is why 38% of sequences were labelled "reentrant".
The classification was an artifact, not a finding.

**3. c_sat was quantised, and 18% of it is physically impossible.** The grid stepped
in 25.6 uM increments, but 55% of records sit below 50 uM and the 25th percentile is
8 uM — so 400 of 623 values landed on a single grid point. Worse, **154 sequences
(18%) have c_sat = 0**: the model predicts phase separation at *zero* protein
concentration.

Monotonicity does not fix that. It forces the curve not to fall, but says nothing
about where it starts. The missing ingredient is a boundary condition:
**P(LLPS) -> 0 as concentration -> 0.** No protein, no droplets.

This notebook fixes the grids and adds that boundary condition.

**Setup:** T4 GPU, ~25 min. Upload `llpsense_final_noseq.csv` and `llpsense_t5.npz`.

## 1. Setup

In [ ]:
import numpy as np, pandas as pd, subprocess, time, json, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from scipy import stats
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0

from google.colab import files
up = files.upload()
df  = pd.read_csv("llpsense_final_noseq.csv")
z   = np.load("llpsense_t5.npz", allow_pickle=True)
emb = pd.DataFrame(z["X"], index=z["SeqID"])
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y, w = df.Label.to_numpy(), df.w_record.to_numpy()
N_EMB = emb.shape[1]; IDX = {c: N_EMB+i for i,c in enumerate(COND)}
UP = ['Conc','PEG300-1k','PEG3k-6k','PEG8k-20k','Ficoll','Dextran -40','Dextran 70-']
MONO = tuple([0]*N_EMB + [1 if c in UP else 0 for c in COND])
MAXV = dict(Conc=1000., NaCl=2000., temp=60., pH=14.)
HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899, random_state=42)
RUNTIME = dict(tree_method="hist", eval_metric="logloss",
               **({"device":"cuda"} if GPU else {"n_jobs":-1}))
print("ready", X.shape)

## 2. Where does the data actually live?

Any sweep outside this range is extrapolation, and tree models extrapolate as a
constant. This figure defines the honest sweep limits for everything that follows.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, c in zip(ax, ["Conc", "NaCl"]):
    v = df[c].to_numpy() * MAXV[c]
    a.hist(v, bins=60, color="C0")
    p95 = np.percentile(v, 95)
    a.axvline(p95, color="crimson", ls="--", label=f"95th pct = {p95:.0f}")
    a.set(xlabel=f"{c} ({'uM' if c=='Conc' else 'mM'})", ylabel="rows", yscale="log")
    a.legend()
plt.suptitle("Data support — sweeps beyond the dashed line are extrapolation")
plt.tight_layout(); plt.savefig("data_support.png", dpi=150); plt.show()

SUP = {c: np.percentile(df[c]*MAXV[c], 95) for c in ["Conc", "NaCl"]}
print({k: round(v,1) for k,v in SUP.items()})

## 3. Boundary-anchored model

A physically trivial fact the data never states: at zero protein there is no phase
separation. We add it explicitly, by duplicating real condition rows with
concentration set to 0 and label 0, at modest total weight.

Combined with monotonicity this guarantees the curve starts low and rises once,
making c_sat well defined for every sequence.

In [ ]:
from xgboost import XGBClassifier

def anchored(X, y, w, frac=0.10, seed=0):
    """append zero-concentration negative anchors carrying `frac` of total weight"""
    rng = np.random.default_rng(seed)
    pick = rng.choice(len(X), size=int(0.3*len(X)), replace=False)
    Xa = X[pick].copy(); Xa[:, IDX['Conc']] = 0.0
    ya = np.zeros(len(Xa), dtype=int)
    wa = np.full(len(Xa), frac*w.sum()/len(Xa))
    return np.vstack([X, Xa]), np.concatenate([y, ya]), np.concatenate([w, wa])

Xa, ya, wa = anchored(X, y, w)
print(f"training rows: {len(X):,} -> {len(Xa):,} (anchors carry 10% of total weight)")

models = {}
models["constrained"] = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(X, y, sample_weight=w)
models["constrained + boundary"] = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(Xa, ya, sample_weight=wa)
print("trained")

## 4. c_sat on a log grid, with and without the anchor

Log spacing because LLPS concentrations span orders of magnitude; the previous linear
grid put 400 of 623 sequences on one point.

In [ ]:
rep  = df.groupby("SeqID").head(1).index.to_numpy()
Xr   = X[rep].copy()
names = df.loc[rep, "ProteinName"].to_numpy()
cgrid_uM = np.logspace(np.log10(0.5), np.log10(SUP["Conc"]), 60)
cgrid    = cgrid_uM / MAXV["Conc"]

def csat_of(model):
    P = np.zeros((len(Xr), len(cgrid)))
    for k, v in enumerate(cgrid):
        Xt = Xr.copy(); Xt[:, IDX['Conc']] = v
        P[:, k] = model.predict_proba(Xt)[:, 1]
    cs = np.full(len(Xr), np.nan)
    for i in range(len(Xr)):
        idx = np.where(P[i] > 0.5)[0]
        if len(idx): cs[i] = cgrid_uM[idx[0]]
    return P, cs

out = {}
for nm, m in models.items():
    P, cs = csat_of(m); out[nm] = (P, cs)
    at_floor = np.nansum(cs <= cgrid_uM[0]*1.001)
    print(f"{nm:26s} c_sat defined {np.isfinite(cs).sum():3d}/{len(cs)}  "
          f"at grid floor (unphysical) {at_floor:3d} ({100*at_floor/len(cs):.0f}%)  "
          f"median {np.nanmedian(cs):6.1f} uM  unique values {len(np.unique(np.round(cs[np.isfinite(cs)],2)))}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, (nm, (P, cs)) in zip(ax, out.items()):
    a.hist(cs[np.isfinite(cs)], bins=np.logspace(np.log10(0.5), np.log10(SUP["Conc"]), 35),
           color="C2", edgecolor="w")
    a.set(xscale="log", xlabel="c_sat (uM)", ylabel="sequences", title=nm)
plt.tight_layout(); plt.savefig("csat_logfixed.png", dpi=150); plt.show()

plt.figure(figsize=(6,4.5))
for nm, (P, cs) in out.items():
    plt.plot(cgrid_uM, P.mean(0), lw=2, label=nm)
plt.axhline(.5, color="k", ls=":", lw=1)
plt.xscale("log"); plt.xlabel("protein concentration (uM)"); plt.ylabel("mean P(LLPS)")
plt.title("Mean concentration response"); plt.legend(); plt.tight_layout()
plt.savefig("conc_response.png", dpi=150); plt.show()

## 5. Salt reentrance, done honestly

Two fixes: sweep only within the data support, and require the interior peak to
**stand out** from both ends by a real margin before calling it reentrant. A flat
curve is now classified as flat, not reentrant.

In [ ]:
sgrid_mM = np.linspace(0, SUP["NaCl"], 30)
sgrid    = sgrid_mM / MAXV["NaCl"]
m = models["constrained + boundary"]
Pn = np.zeros((len(Xr), len(sgrid)))
for k, v in enumerate(sgrid):
    Xt = Xr.copy(); Xt[:, IDX['NaCl']] = v
    Pn[:, k] = m.predict_proba(Xt)[:, 1]

PROM = 0.05                                  # required prominence
pk   = Pn.argmax(1)
prom = Pn.max(1) - np.maximum(Pn[:, 0], Pn[:, -1])
rng_ = Pn.max(1) - Pn.min(1)
shape = np.where(rng_ < PROM, "flat (no salt response)",
          np.where((pk > 1) & (pk < len(sgrid)-2) & (prom >= PROM), "reentrant (true interior peak)",
            np.where(Pn[:,-1] < Pn[:,0], "monotonic decreasing", "monotonic increasing")))
print(pd.Series(shape).value_counts().to_string())
print(f"\nprevious run called 323/854 reentrant using argmax with no prominence test")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(sgrid_mM, Pn.mean(0), lw=2.5)
ax[0].fill_between(sgrid_mM, np.percentile(Pn,25,0), np.percentile(Pn,75,0), alpha=.2)
ax[0].set(xlabel="NaCl (mM)", ylabel="mean P(LLPS)", title="Salt response within data support")
sel = np.where(shape == "reentrant (true interior peak)")[0][:6]
for s in sel: ax[1].plot(sgrid_mM, Pn[s], lw=1.6, label=str(names[s])[:16])
ax[1].set(xlabel="NaCl (mM)", ylabel="P(LLPS)", title=f"Reentrant profiles (prominence > {PROM})")
if len(sel): ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("salt_reentrance_fixed.png", dpi=150); plt.show()

pd.DataFrame({"SeqID": df.loc[rep,"SeqID"].to_numpy(), "ProteinName": names,
              "c_sat_uM": out["constrained + boundary"][1], "salt_shape": shape}
             ).to_csv("csat_v2.csv", index=False)

## 6. What does the boundary condition cost?

Same paired design as before: identical splits, anchors added to the training fold
only, never to the test fold.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

for rname, key in [("A  new condition","RecordID"), ("B  new protein","SeqID")]:
    g, du, dc = df[key].to_numpy(), [], []
    for s in [1, 2]:
        for i, j in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(X, y, groups=g):
            if len(np.unique(y[j])) < 2: continue
            m0 = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(X[i], y[i], sample_weight=w[i])
            Xi, yi, wi = anchored(X[i], y[i], w[i])
            m1 = XGBClassifier(**HP, monotone_constraints=MONO, **RUNTIME).fit(Xi, yi, sample_weight=wi)
            du.append(roc_auc_score(y[j], m0.predict_proba(X[j])[:,1], sample_weight=w[j]))
            dc.append(roc_auc_score(y[j], m1.predict_proba(X[j])[:,1], sample_weight=w[j]))
    du, dc = np.array(du), np.array(dc); d = dc - du
    t, p = stats.ttest_rel(dc, du)
    print(f"{rname:20s} constrained {du.mean():.4f} -> +boundary {dc.mean():.4f}  "
          f"diff {d.mean():+.4f}  t {t:+.2f}  p {p:.3f}", flush=True)

from google.colab import files
for f in ["csat_v2.csv","csat_logfixed.png","conc_response.png",
          "salt_reentrance_fixed.png","data_support.png"]:
    files.download(f)

### What this run decides

- **c_sat spread over a real range** instead of piling on one grid point -> usable for
  ranking simulation targets.
- **Unphysical c_sat ~ 0 cases driven to zero** by the boundary anchor -> the model now
  respects both monotonicity *and* the correct limit at zero concentration.
- **An honest reentrant fraction.** Expect it to be far below the earlier 38%. If it
  collapses to near zero, say so: it would mean the model has not learned reentrance,
  which is a legitimate negative result given that salt was left unconstrained.
- **The cost of the boundary condition**, measured the same paired way as before.